# CCT College Dublin - Continuous Assessment 1 (CA1)

**Student Name:** Daniel Adejare

**Student ID:** 2026848

---

| Field | Details |
| :--- | :--- |
| **Programme Title** | BSc Computing and IT (Stage 4, Semester 1) |
| **Cohort Details** | DataVisComms_BScCompIT_Sept23 FT Stage 4 Semester 1 |
| **Module Title** | Data Visualisation and Communication |
| **Lecturer** | Taufique Ahmed |
| **Assignment Title** | CA1 (Individual, 40% Weighting) |
| **Submission Deadline** | 07-11-2026 |

---

### Mandatory Submission Links
* **Screen Capture Video (Google Drive):** [Pending Video Link]

* **GitHub Repository:** https://github.com/Dee-17-C/CA1_Data_Visualisation


---

### Overview
This notebook presents **7 distinct visualisation types** analysing US employment and compensation across Artificial Intelligence (AI), Data Science, and Software Development roles using the May 2023 Bureau of Labor Statistics (BLS) OEWS dataset (U.S. Bureau of Labor Statistics, 2024), alongside multi-year datasets (2021–2023) for the longitudinal Challenge task.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

# Target SOC codes for AI, Data Science & Software Development
TARGET_ROLES = {
    '15-2051': 'Data Scientists',
    '15-1252': 'Software Developers',
    '15-1221': 'Computer & Info Research Scientists (AI)',
    '15-1253': 'Software QA Analysts & Testers',
    '15-1243': 'Database Architects',
    '15-1211': 'Computer Systems Analysts',
    '15-2031': 'Operations Research Analysts',
    '15-1299': 'Computer Occupations, All Other'
}

# Load May 2023 dataset
file_2023 = 'all_data_M_2023.xlsx' if os.path.exists('all_data_M_2023.xlsx') else 'oesm23all/all_data_M_2023.xlsx'
df_raw = pd.read_excel(file_2023, sheet_name=0)
df_raw.columns = df_raw.columns.str.upper()

# Convert BLS suppression symbols ('*', '**', '#') to NaN and cast to float
numeric_cols = [
    'TOT_EMP', 'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT',
    'A_MEAN', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90'
]

def clean_bls_numeric(df, cols):
    df_clean = df.copy()
    for col in cols:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(
                df_clean[col].astype(str).str.replace(',', '').replace({'*': np.nan, '**': np.nan, '#': np.nan}),
                errors='coerce'
            )
    return df_clean

df_2023 = clean_bls_numeric(df_raw, numeric_cols)
df_2023['ROLE_NAME'] = df_2023['OCC_CODE'].map(TARGET_ROLES)

# Filter for National Cross-Industry, Cross-Ownership totals (AREA_TYPE == 1, NAICS == '000000', OWN_CODE == 1235)
df_nat_2023 = df_2023[
    (df_2023['AREA_TYPE'] == 1) &
    (df_2023['NAICS'].astype(str).str.startswith('000000')) &
    (df_2023['OWN_CODE'] == 1235) &
    (df_2023['OCC_CODE'].isin(TARGET_ROLES.keys()))
].copy()

df_nat_2023[['OCC_CODE', 'ROLE_NAME', 'TOT_EMP', 'A_MEAN', 'A_MEDIAN']].sort_values(by='TOT_EMP', ascending=False)